# CellDRIFT

Implementation from the [original authors' release](https://github.com/MorganLevineLab/CellDRIFT/tree/066b3e816795f3c0b372ff8d33f59185a490bd8b), accompanying [Minteer et al. (2023)](https://doi.org/10.1126/sciadv.adf4163). CellDRIFT measures replicative history calibrated to cumulative population doublings.

The 2,025 yellow and 297 tan CpGs enter separate PCA projections, followed by a 62-PC elastic-net score. We fold both linear projections and centering into an equivalent 2,322-CpG linear layer, at the authors' exact `lambda.min`. Missing columns use module means; supplied NaNs become zero, as in the original R function.

Build-only dependency: `uv run --with rdata jupyter lab`. Upstream GitHub DESCRIPTION has no license value; the [authors' Zenodo record](https://doi.org/10.5281/zenodo.7693699) lists Other (Open). No stronger weight license is asserted.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import tempfile
import os
import pyaging as pya

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "clocks/build_replication.py").is_file())
spec = importlib.util.spec_from_file_location("build_replication", root / "clocks/build_replication.py")
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)
source = Path(os.environ.get("PYAGING_CELLDRIFT_SOURCE", str(Path(tempfile.gettempdir()) / "CellDRIFT-original")))
if not source.exists():
    subprocess.run(["git", "clone", "https://github.com/MorganLevineLab/CellDRIFT", str(source)], check=True)
    subprocess.run(["git", "-C", str(source), "checkout", builder.CELLDRIFT_COMMIT], check=True)
model = builder.build_celldrift(source)

In [2]:
model.metadata['clock_name'] = 'celldrift'
model.metadata['data_type'] = 'DNA methylation'  # Paper: DNA methylation
model.metadata['species'] = 'Homo sapiens'  # Paper: immortalized human cells
model.metadata['year'] = 2023
model.metadata['approved_by_author'] = '⌛'
model.metadata['citation'] = 'Minteer, C. J., et al. (2023). More than bad luck: Cancer and aging are linked to replication-driven changes to the epigenome. Science Advances, 9(29), eadf4163.'
model.metadata['doi'] = 'https://doi.org/10.1126/sciadv.adf4163'
model.metadata['notes'] = 'Replication signature trained on 31 serially passaged hTERT-immortalized human astrocyte samples. The released 62-PC elastic-net predictor over 2,322 CpGs is folded algebraically into a single linear score. Output is calibrated to cumulative population doublings, not years. Original-author missingness is preserved: supplied NaNs become zero, whereas absent CpGs use module means. GitHub commit 066b3e816795f3c0b372ff8d33f59185a490bd8b; DOI 10.5281/zenodo.7693699. Upstream software license is unspecified (Zenodo: Other (Open)).'
model.metadata['research_only'] = None
model.metadata['tissue'] = ['cultured human cells']  # Paper: hTERT training data [this study (GSE226079), n = 31 samples]
model.metadata['predicts'] = ['mitotic age']  # Paper: epigenetic replication signature
model.metadata['training_target'] = ['population doublings']  # Paper: cumulative popultion doubling events (cPD) of cultured human cells
model.metadata['unit'] = ['population doublings']  # Paper: cumulative popultion doubling events (cPD)
model.metadata['model_type'] = 'PCA + elastic net regression'  # Paper: Combine PC sets; Use the elastic net objects to predict values from the final PC object
model.metadata['platform'] = ['Illumina EPIC']  # Paper: Infinium HumanMethylationEPIC BeadChip
model.metadata['population'] = 'human cell cultures'  # Paper: cultured human cells
model.metadata['journal'] = 'Science Advances'
model.metadata['last_author'] = 'Morgan E. Levine'
model.metadata['n_features'] = 2322
model.metadata['citations'] = 29
model.metadata['citations_date'] = '2026-10-02'
model.version = "0.5.4"

Original-author R reference fixtures cover nine author-supplied samples, supplied NaNs, absent probes, beta boundaries, and batch sizes 1, 3, and 64. Regenerate the R outputs with `tests/data/replication/generate_references.R`; run `uv run pytest tests/models/test_replication.py` to compare with pyaging.

In [3]:
builder.save_models([model], root / "clocks/weights", root / "clocks/metadata/celldrift_0.5.4.pt")
print(model.metadata["clock_name"], len(model.features))

celldrift 2322
